In [1]:
import torch.nn as nn
import torchvision
import torch
from torchinfo import summary
from DWT import DWT_2D
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import ast
from scipy import stats
from tqdm import tqdm
import time

import os
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix, f1_score
import seaborn as sns

from IXCM import IXCNet
from Loss import bce_loss, var_loss

In [2]:
LABEL_COLUMNS = [
    'Consolidation', 'Edema', 'Cardiomegaly', 'Pleural Effusion', 'Pneumonia'
]

class MIMICXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None, class_name=['Atelectasis']):
        self.dataframe = dataframe
        self.transform = transform
        self.labels = torch.tensor(self.dataframe[class_name].values, dtype=torch.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_paths_list = self.dataframe.iloc[idx]['image_paths_list']
        labels = self.labels[idx]

        selected_path = None

        pa_paths = [p for p in image_paths_list if 'PA' in p]
        
        if len(pa_paths) > 0:
            selected_path = pa_paths[0]
        else:
            ap_paths = [p for p in image_paths_list if 'AP' in p]
            if len(ap_paths) > 0:
                selected_path = ap_paths[0]
            else:
                selected_path = image_paths_list[0]

        image = Image.open(selected_path).convert('L')

        if self.transform:
            image = self.transform(image)

        return image, labels

In [3]:
def train_one_epoch(model, dataloader, criterion_bce, criterion_var, optimizer, device, gc=0.01, loss_mode='bcevar', var_weight=1.0, img_dec=True):
    model.train()

    run_loss = 0.0
    run_bce = 0.0
    run_var = 0.0
    
    preds_list = []
    labels_list = []

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        alphas, betas, outs = model(inputs)

        loss_bce = criterion_bce(outs, labels)
            
        if loss_mode == 'bcevar':
            if img_dec:
                alpha_s = alphas[:, 0::2]  
                beta_s  = betas[:, 0::2]
                alpha_a = alphas[:, 1::2]
                beta_a  = betas[:, 1::2]
                loss_var = criterion_var(alpha_s, beta_s, alpha_a, beta_a, labels)
            else:
                alpha_s = alphas
                beta_s  = betas
                loss_var = criterion_var(alpha_s, beta_s, alpha_s, beta_s, labels)
            loss = loss_bce + var_weight * loss_var
        else:
            loss = loss_bce

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gc)
        optimizer.step()

        bs = inputs.size(0)
        run_loss += loss.item() * bs
        run_bce  += loss_bce.item() * bs
        if loss_mode == 'bcevar':
            run_var  += loss_var.item() * bs

        preds_list.extend(outs.detach().cpu().numpy())
        labels_list.extend(labels.cpu().numpy())

    dataset_size = len(dataloader.dataset)
    epoch_loss = run_loss / dataset_size
    epoch_bce  = run_bce / dataset_size
    epoch_var  = run_var / dataset_size

    preds_arr = np.array(preds_list)
    labels_arr = np.array(labels_list)
    preds_bin = preds_arr > 0.5

    correct_prediction = (preds_bin == labels_arr)
    class_accs = np.mean(correct_prediction, axis=0)
    acc = np.mean(class_accs)

    try:
        auc = roc_auc_score(labels_arr, preds_arr, average='macro')
    except ValueError:
        auc = 0.5 

    return epoch_loss, epoch_bce, epoch_var, acc, auc

def validate(model, dataloader, criterion_bce, criterion_var, device, loss_mode='bcevar', var_weight=1.0, img_dec=True):
    model.eval()

    run_loss = 0.0
    run_bce = 0.0
    run_var = 0.0
    
    preds_list = []
    labels_list = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            alphas, betas, outs = model(inputs)

            loss_bce = criterion_bce(outs, labels)
            
            if loss_mode=='bcevar':
                if img_dec:
                    alpha_s = alphas[:, 0::2]  
                    beta_s  = betas[:, 0::2]
                    alpha_a = alphas[:, 1::2]
                    beta_a  = betas[:, 1::2]
                    loss_var = criterion_var(alpha_s, beta_s, alpha_a, beta_a, labels)
                else:
                    alpha_s = alphas
                    beta_s  = betas
                    loss_var = criterion_var(alpha_s, beta_s, alpha_s, beta_s, labels)
                loss = loss_bce + var_weight * loss_var
            else:
                loss = loss_bce

            bs = inputs.size(0)
            run_loss += loss.item() * bs
            run_bce  += loss_bce.item() * bs
            if loss_mode == 'bcevar':
                run_var  += loss_var.item() * bs

            preds_list.extend(outs.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    dataset_size = len(dataloader.dataset)
    epoch_loss = run_loss / dataset_size
    epoch_bce  = run_bce / dataset_size
    epoch_var  = run_var / dataset_size
    
    preds_arr = np.array(preds_list)
    labels_arr = np.array(labels_list)
    preds_bin = preds_arr > 0.5

    correct_prediction = (preds_bin == labels_arr)
    class_accs = np.mean(correct_prediction, axis=0)
    acc = np.mean(class_accs)
    try:
        auc = roc_auc_score(labels_arr, preds_arr, average='macro')
    except ValueError:
        auc = 0.5

    return epoch_loss, epoch_bce, epoch_var, acc, auc, labels_arr, preds_bin

In [4]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pth', trace_func=print):

        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

def train_model_with_strategy(
    model, 
    train_loader, 
    val_loader, 
    criterion_bce, 
    criterion_var, 
    device,
    epochs=50,
    lr_start=1e-3, 
    lr_end=1e-6,
    warmup_epochs=5,
    use_early_stopping=True,
    patience=10,
    use_scheduler=True,
    weight_decay=0.01,
    gradient_clip=3.0,
    loss_mode='bcevar',
    var_weight=1.0,
    save_dir='./checkpoints',
    img_dec=True
):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    best_acc = 0.0
    best_auc = 0.0
    best_loss = 1e8
    best_bce = 1e8
    best_var = 1e8
    optimizer = optim.Adam(model.parameters(), lr=lr_start, weight_decay=weight_decay)

    scheduler = None
    if use_scheduler:
        scheduler_epochs = epochs - warmup_epochs
        if scheduler_epochs > 0:
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=scheduler_epochs, eta_min=lr_end)
    
    early_stopping = EarlyStopping(patience=patience, verbose=False, path='best_model_strategy.pth')

    history = {
        'train': {'loss': [], 'bce': [], 'var': [], 'acc': [], 'auc': []},
        'val':   {'loss': [], 'bce': [], 'var': [], 'acc': [], 'auc': []},
        'lr': []
    }

    print(f"{'='*100}")
    print(f"Start Training: Total Epochs={epochs}")
    if use_scheduler:
        print(f"Strategy: Warmup ({warmup_epochs} epochs @ {lr_end:.1e}) -> CosineAnnealing ({lr_start:.1e}->{lr_end:.1e})")
    else:
        print(f"Strategy: Constant Learning Rate ({lr_start:.1e})")
    print(f"{'='*100}")
    for epoch in range(epochs):
        epoch_start_time = time.time()
        if use_scheduler:
            if epoch < warmup_epochs:
                current_lr = lr_end
                for param_group in optimizer.param_groups:
                    param_group['lr'] = current_lr
                phase_name = "Warmup"
            else:
                if epoch == warmup_epochs:
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = lr_start
                
                current_lr = optimizer.param_groups[0]['lr']
                phase_name = "Anneal"
        else:
            current_lr = optimizer.param_groups[0]['lr'] 
            phase_name = "Constant"

        t_loss, t_bce, t_var, t_acc, t_auc = train_one_epoch(
            model, train_loader, criterion_bce, criterion_var, optimizer, device, gc=gradient_clip, loss_mode=loss_mode, var_weight=var_weight, img_dec=img_dec
        )

        v_loss, v_bce, v_var, v_acc, v_auc, _, _ = validate(
            model, val_loader, criterion_bce, criterion_var, device, loss_mode=loss_mode, var_weight=var_weight, img_dec=img_dec
        )

        if use_scheduler and scheduler is not None and epoch >= warmup_epochs:
            scheduler.step()
            
        epoch_end_time = time.time()
        epoch_duration = epoch_end_time - epoch_start_time
        epoch_mins = int(epoch_duration // 60)
        epoch_secs = int(epoch_duration % 60)

        history['train']['loss'].append(t_loss)
        history['train']['bce'].append(t_bce)
        history['train']['var'].append(t_var)
        history['train']['acc'].append(t_acc)
        history['train']['auc'].append(t_auc)
        
        history['val']['loss'].append(v_loss)
        history['val']['bce'].append(v_bce)
        history['val']['var'].append(v_var)
        history['val']['acc'].append(v_acc)
        history['val']['auc'].append(v_auc)
        history['lr'].append(current_lr)

        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] [{phase_name}] LR: {current_lr:.2e} Time: {epoch_mins}m {epoch_secs}s")
        print(f"  Train | Loss: {t_loss:.4f} (BCE: {t_bce:.4f}, Var: {t_var:.4f}) | Acc: {t_acc:.4f} | AUC: {t_auc:.4f}")
        print(f"  Val   | Loss: {v_loss:.4f} (BCE: {v_bce:.4f}, Var: {v_var:.4f}) | Acc: {v_acc:.4f} | AUC: {v_auc:.4f}")

        if v_loss < best_loss:
            best_loss = v_loss
            save_path_loss = os.path.join(save_dir, 'best_model_loss.pth')
            torch.save(model.state_dict(), save_path_loss)
            print(f"  -> [Saved] Best Loss updated: {best_loss:.4f}")

        if v_bce < best_bce:
            best_bce = v_bce
            save_path_bce = os.path.join(save_dir, 'best_model_bce.pth')
            torch.save(model.state_dict(), save_path_bce)
            print(f"  -> [Saved] Best BCE updated: {best_bce:.4f}")

        if v_var < best_var:
            best_var = v_var
            save_path_var = os.path.join(save_dir, 'best_model_var.pth')
            torch.save(model.state_dict(), save_path_var)
            print(f"  -> [Saved] Best VAR updated: {best_var:.4f}")
            
        if v_acc > best_acc:
            best_acc = v_acc
            save_path_acc = os.path.join(save_dir, 'best_model_acc.pth')
            torch.save(model.state_dict(), save_path_acc)
            print(f"  -> [Saved] Best Acc updated: {best_acc:.4f}")

        if v_auc > best_auc:
            best_auc = v_auc
            save_path_auc = os.path.join(save_dir, 'best_model_auc.pth')
            torch.save(model.state_dict(), save_path_auc)
            print(f"  -> [Saved] Best AUC updated: {best_auc:.4f}")
            
        print("-" * 100)

        if use_early_stopping:
            early_stopping(v_loss, model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}!")
                break
    
    if use_early_stopping:
        print("Loading best model weights...")
        model.load_state_dict(torch.load('best_model_strategy.pth'))

    return model, history

In [5]:
def plot_loss_curve(train_losses, val_losses, save_path='./loss_curve.png', metric_name='Loss') :
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label=f'Train {metric_name}')
    plt.plot(val_losses, label=f'Val {metric_name}')
    plt.title(f'Training and Validation {metric_name}')
    plt.xlabel('Epochs')
    if metric_name == "AUC" or metric_name == "ACC":
        plt.ylim([0, 1.1])
    plt.ylabel(f'{metric_name}')
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)

In [6]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random Seed set to: {seed}")

In [ ]:
for LABEL_COLUMN in LABEL_COLUMNS:
    img_dec=False
    loss_mode = 'bce'
    class_name = [LABEL_COLUMN]
    seed_everything(42)
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if img_dec:
        BATCH_SIZE = 64
    else:
        BATCH_SIZE = 32
    stid_imgpath_csv_path = f''
    stid_imgpath_df = pd.read_csv(stid_imgpath_csv_path)
    train_val_test_split_csv_path = ''
    split_df = pd.read_csv(train_val_test_split_csv_path)
    
    train_study_ids = split_df[split_df['split'] == 'train']['study_id'].unique()
    val_study_ids = split_df[split_df['split'] == 'validate']['study_id'].unique()
    test_study_ids = split_df[split_df['split'] == 'test']['study_id'].unique()
    
    train_df = stid_imgpath_df[stid_imgpath_df['study_id'].isin(train_study_ids)].copy()
    val_df = stid_imgpath_df[stid_imgpath_df['study_id'].isin(val_study_ids)].copy()
    test_df = stid_imgpath_df[stid_imgpath_df['study_id'].isin(test_study_ids)].copy()
    
    def parse_and_fix_paths(path_str):
        if isinstance(path_str, list):
            path_list = path_str
        else:
            path_list = ast.literal_eval(path_str)
        OLD_ROOT = ''
        NEW_ROOT = ''
        
        return [p.replace(OLD_ROOT, NEW_ROOT) for p in path_list]
    
    train_df['image_paths_list'] = train_df['image_paths_list'].apply(parse_and_fix_paths)
    val_df['image_paths_list'] = val_df['image_paths_list'].apply(parse_and_fix_paths)
    test_df['image_paths_list'] = test_df['image_paths_list'].apply(parse_and_fix_paths)
    
    train_transforms = transforms.Compose([
        transforms.Resize(540),
        transforms.RandomCrop(512),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    
    val_transforms = transforms.Compose([
        transforms.Resize(540),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    
    train_dataset = MIMICXrayDataset(train_df, transform=train_transforms, class_name=class_name)
    val_dataset = MIMICXrayDataset(val_df, transform=val_transforms, class_name=class_name)
    test_dataset = MIMICXrayDataset(test_df, transform=val_transforms, class_name=class_name)
    
    print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")
    
    def worker_init_fn(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed + worker_id)
        random.seed(worker_seed + worker_id)
        
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True,
                              persistent_workers=True, prefetch_factor=2, worker_init_fn=worker_init_fn)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True,
                            persistent_workers=True, prefetch_factor=2, worker_init_fn=worker_init_fn)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True,
                             persistent_workers=True, prefetch_factor=2, worker_init_fn=worker_init_fn)

    result_root_path = f''
    weight_decay = 0.0001
    gradient_clip = 1.0
    var_weight = 1.0
    KL_lmd = 0.1
    lr=0.0005
    result_root_path = result_root_path + '_wd' + str(weight_decay).replace('.', '_') + '_gc' + str(gradient_clip).replace('.', '_') + '_vw' + str(var_weight).replace('.', '_') + '_lm_' + loss_mode + '_lr_' + str(lr) + '_kllmd_' + str(KL_lmd) + '_imgdec_' + str(int(img_dec))
    STRATEGY_PARAMS = {
        'epochs': 80,
        'lr_start': lr,
        'lr_end': 1e-4,
        'use_early_stopping': False,
        'patience': 10,
        'warmup_epochs': 0,
        'use_scheduler': True,
        'weight_decay': weight_decay,
        'gradient_clip': gradient_clip,
        'loss_mode': loss_mode,
        'var_weight': var_weight,
        'save_dir': result_root_path,
        'img_dec': img_dec
    }

    model = IXCNet(evi_vec_dim=64, wavename='haar', ydim=1, in_channels=1, img_dec=img_dec).to(DEVICE)
    criterion_bce = bce_loss(reduction='mean')
    criterion_var = var_loss(reduction='mean', lmd=KL_lmd, img_dec=img_dec)

    final_model, history = train_model_with_strategy(
        model, 
        train_loader, 
        val_loader, 
        criterion_bce, 
        criterion_var, 
        DEVICE,
        **STRATEGY_PARAMS
    )
    
    history_path = os.path.join(result_root_path, 'history.npy')
    np.save(history_path, history)

    plot_loss_curve(history['train']['loss'], history['val']['loss'], save_path=result_root_path + '/loss_history.png', metric_name="Loss")
    plot_loss_curve(history['train']['bce'], history['val']['bce'], save_path=result_root_path + '/bce_history.png', metric_name="BCE")
    plot_loss_curve(history['train']['var'], history['val']['var'], save_path=result_root_path + '/var_history.png', metric_name="VAR")
    plot_loss_curve(history['train']['acc'], history['val']['acc'], save_path=result_root_path + '/acc_history.png', metric_name="ACC")
    plot_loss_curve(history['train']['auc'], history['val']['auc'], save_path=result_root_path + '/auc_history.png', metric_name="AUC")